In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import time
import torch.nn.functional as F
import timm
from sklearn.metrics import accuracy_score, classification_report

IMAGE_SIZE = 224 

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


test_ds      = datasets.ImageFolder("test", transform=transform)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


num_classes = len(test_ds.classes) 

model_mnv3 = models.mobilenet_v3_large(pretrained=False)
model_mnv3.classifier[3] = nn.Linear(
    model_mnv3.classifier[3].in_features,
    num_classes
)

model_mnv3.load_state_dict(torch.load("models/mobilenet_v3large.pth", map_location=device))
model_mnv3 = model_mnv3.to(device)
model_mnv3.eval()


predictionresult = []
def inference(model1, loader):
    y_true, y_pred = [], []
    class_names = loader.dataset.classes  
    global_index = 0  
    with torch.no_grad():
        for i,(imgs, labels) in enumerate(loader):
            imgs = imgs.to(device)
            labels = labels.to(device)
            out1 = model1(imgs)
            prob1 = F.softmax(out1, dim=1)
            preds = prob1.argmax(dim=1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    return y_true, y_pred


y_true, y_pred = inference(
    model_mnv3,
    test_loader
)
acc = accuracy_score(y_true, y_pred)
print("Accuracy:", acc)

print(classification_report(
    y_true,
    y_pred,
    target_names=test_ds.classes
))


/home/ai_atrlbcau/miniconda3/envs/tensor_torch/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/ai_atrlbcau/miniconda3/envs/tensor_torch/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Ensemble Accuracy: 0.8406805877803558
                  precision    recall  f1-score   support

       Argulosis       1.00      1.00      1.00       189
             EUS       0.94      0.89      0.91       179
Fin and Tail Rot       0.69      0.93      0.79       180
        Gill Rot       0.95      0.69      0.80       181
      Lernaeosis       0.90      0.65      0.75       186
      Septicemia       0.71      0.99      0.83       191
     cotton wool       0.86      0.73      0.79       187

        accuracy                           0.84      1293
       macro avg       0.86      0.84      0.84      1293
    weighted avg       0.86      0.84      0.84      1293

